# HydroSeason: Rainfall Context

**What this shows:** attaching rainfall as ancillary context to the same
water-only workflow from [01_quickstart.ipynb](01_quickstart.ipynb), and
confirming rainfall never changes the water regime, route, or boundaries —
only the report gains a rainfall-comparison panel.

**Requirements:** core install only (`pip install hydroseason`).

**Network:** none. Uses a committed case-study rainfall CSV
(`rainfall_csv_path=`) instead of fetching SILO live
(`fetch_rainfall=True`, which needs network — see the note at the end).

**Runtime:** a few seconds.

## 1. Run with rainfall attached

Same catchment, same extent CSV, same `analysis_options` as the quickstart
notebook — the only addition is `rainfall_csv_path`.

In [1]:
from pathlib import Path

from hydroseason import run_hydroseason

CASE_STUDY_CSV = Path("../case_studies/data/extent/fitzroy_river_wa_30m.csv")
RAINFALL_CSV = Path("../case_studies/data/rainfall/fitzroy_river_wa_silo_rainfall.csv")

result = run_hydroseason(
    CASE_STUDY_CSV,
    output_dir="output/03_rainfall_context",
    aoi_name="Fitzroy River (WA)",
    analysis_options={"phase_model": "rule_based"},
    rainfall_csv_path=RAINFALL_CSV,
)

print(f"Regime: {result.analysis.regime.regime}")
print(f"Route: {result.analysis.route}")
print(f"Rainfall status: {result.rainfall_status}")

Regime: seasonal
Route: per_year_detection
Rainfall status: provided


## 2. Rainfall is additive, never authoritative

`result.rainfall_comparison` shows what rainfall adds: its own regime
assessment, and how it compares to the water-only regime above
(`divergence`, `peak_lag_months`). None of this can feed back into
`result.analysis` — that was already finalized from water alone before
rainfall was even loaded.

In [2]:
comparison = result.rainfall_comparison

print(f"Water regime:    {comparison.extent.regime} (SNR {comparison.extent.amplitude_snr:.2f})")
print(f"Rainfall regime: {comparison.rainfall.regime} (SNR {comparison.rainfall.amplitude_snr:.2f})")
print(f"Divergence:      {comparison.divergence}")
print(f"Peak lag:        {comparison.peak_lag_months} month(s)")
print()
print(comparison.interpretation)

Water regime:    seasonal (SNR 3.64)
Rainfall regime: seasonal (SNR 5.71)
Divergence:      agree
Peak lag:        1 month(s)

Both rainfall and observed surface-water extent show a reproducible annual cycle. Peak lags rainfall by 1 month(s). Water availability here plausibly tracks local rainfall seasonality; this is consistent with, not proof of, a direct rainfall-driven relationship.


## 3. Proof: identical route with and without rainfall

Compare against the water-only run from
[01_quickstart.ipynb](01_quickstart.ipynb) — same CSV, same
`analysis_options`, no `rainfall_csv_path`. Regime and route match exactly.

In [3]:
water_only = run_hydroseason(
    CASE_STUDY_CSV,
    output_dir="output/03_rainfall_context_water_only",
    aoi_name="Fitzroy River (WA)",
    analysis_options={"phase_model": "rule_based"},
)

assert water_only.analysis.regime.regime == result.analysis.regime.regime
assert water_only.analysis.route == result.analysis.route
print("Water-only regime/route are unchanged by attaching rainfall:")
print(f"  regime: {water_only.analysis.regime.regime}")
print(f"  route:  {water_only.analysis.route}")

Water-only regime/route are unchanged by attaching rainfall:
  regime: seasonal
  route:  per_year_detection


## 4. Open the report

The rainfall-comparison panel is collapsible in the HTML report — it adds
context without changing the primary water-extent analysis above it. Same
[live example with rainfall](https://tayerthiaggo.github.io/hydroseason/examples/fitzroy-river-wa-rainfall.html)
is linked from the README.

In [4]:
print(f"Open this file in a browser:\n{result.artifacts.html.relative_to(Path.cwd())}")

Open this file in a browser:
output\03_rainfall_context\fitzroy-river-wa.html


## Fetching rainfall live instead

This notebook uses a committed CSV so it runs offline. To fetch SILO
rainfall automatically instead (requires network and `hydroseason[raster]`
for the SILO client), replace `rainfall_csv_path=RAINFALL_CSV` with
`fetch_rainfall=True` — see
[Usage Guide: Add rainfall context](https://tayerthiaggo.github.io/hydroseason/guide/#4-add-rainfall-context-optional).

See also: [Rainfall Context case study](https://tayerthiaggo.github.io/hydroseason/case-studies/rainfall-context/)
for the same proof across all five case-study catchments.